# FixPlanCatalog example

This notebook shows the prototype `FixPlanCatalog`. A catalog combines multiple plan sources so users can query curated plans and generated auto plans through one surface.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from woodpecker.plans import DatasetMatcher, FixPlan, FixRef
from woodpecker.stores import AutoFixPlanStore, FixPlanCatalog, JsonFixPlanStore
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset and a small curated plan store. The curated plan and the auto store can both point to the same underlying units fix.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)

tmp = TemporaryDirectory()
store = JsonFixPlanStore(Path(tmp.name) / "plans.yaml")
store.save_plan(
    FixPlan(
        id="cmip6.curated_units",
        description="Curated CMIP6 units plan",
        match=DatasetMatcher(dataset_id_patterns=["CMIP6.CMIP.*.Amon.tas.*"]),
        steps=[FixRef(id="woodpecker.normalize_tas_units_to_kelvin")],
    )
)

catalog = FixPlanCatalog([store, AutoFixPlanStore()])

The catalog lists plans from all sources and deduplicates by plan id using source order.

In [ ]:
[plan.id for plan in catalog.list_plans()[:8]]

Lookup queries every source. Here the curated CMIP6 plan appears first, followed by the generated one-step auto plan.

In [ ]:
matched_plans = catalog.lookup(dataset)
[plan.id for plan in matched_plans]

The same catalog can resolve plan ids and aliases.

In [ ]:
catalog.get_plan("cmip6.curated_units")

In [ ]:
tmp.cleanup()